# Bab 15: Klasterisasi Partisi dan Berbasis Densitas

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab15/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab15")
print("Folder kerja:", os.getcwd())

## Listing 15.1: K-means dengan tangan dan pengaruh pusat awal

`kode/bab15/kmeans_mini.py`

In [ ]:
import numpy as np

nama = np.array(list("ABCDEFGHIJ"))
X = np.array([[2, 1.0], [3, 1.5], [4, 1.0], [10, 3.0], [12, 3.5],
              [11, 2.5], [6, 6.0], [7, 7.0], [5, 6.5], [8, 6.0]])


def kmeans(pusat, cetak=True):
    for it in range(1, 20):
        d = np.linalg.norm(X[:, None, :] - pusat[None, :, :], axis=2)
        label = d.argmin(1)
        sse = (d.min(1) ** 2).sum()
        baru = np.array([X[label == k].mean(0) for k in range(3)])
        if cetak:
            print(f"iterasi {it}: klaster", "".join(str(l + 1)
                                                    for l in label),
                  f" SSE {sse:.2f}")
            print("  pusat baru:", "  ".join(f"({a:.2f}; {b:.2f})"
                                             for a, b in baru))
        if np.allclose(baru, pusat):
            return label, sse
        pusat = baru


label, sse = kmeans(X[[0, 1, 2]].astype(float))
print("konvergen; SSE akhir", round(sse, 2))
for awal in ["DEF", "AGH"]:
    idx = [list(nama).index(c) for c in awal]
    label, sse = kmeans(X[idx].astype(float), cetak=False)
    print(f"pusat awal {awal}: klaster",
          " ".join("".join(nama[label == k]) for k in range(3)),
          f" SSE {sse:.2f}")

## Listing 15.2: Pengaruh penskalaan pada K-means

`kode/bab15/skala.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab15')
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from siapkan import muat

df, X_mentah = muat(log=False)
_, X_log = muat(log=True)
for nama, X in [("mentah", X_mentah.to_numpy()),
                ("terstandar", StandardScaler().fit_transform(X_mentah)),
                ("log + terstandar", StandardScaler().fit_transform(X_log))]:
    km = KMeans(4, n_init=10, random_state=0).fit(X)
    ari = adjusted_rand_score(df["segmen_laten"], km.labels_)
    ukuran = sorted(np.bincount(km.labels_).tolist() if False else
                    [int((km.labels_ == k).sum()) for k in range(4)])
    print(f"{nama:17s} ARI {ari:.3f}  ukuran klaster {ukuran}")

## Listing 15.3: K-medoids sederhana dan ketahanannya terhadap pencilan

`kode/bab15/kmedoids.py`

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances


def kmedoids(D, k, iterasi=50, seed=0):
    rng = np.random.default_rng(seed)
    med = rng.choice(len(D), k, replace=False)
    for _ in range(iterasi):
        label = D[:, med].argmin(1)
        baru = med.copy()
        for j in range(k):
            anggota = np.where(label == j)[0]
            biaya = D[np.ix_(anggota, anggota)].sum(1)
            baru[j] = anggota[biaya.argmin()]
        if set(baru) == set(med):
            break
        med = baru
    return med, D[:, med].argmin(1)


rng = np.random.default_rng(1)
X = np.vstack([rng.normal([0, 0], 0.5, (30, 2)),
               rng.normal([4, 0], 0.5, (30, 2))])
X_pencilan = np.vstack([X, [[40, 0]]])
for nama, data in [("tanpa pencilan", X), ("dengan pencilan", X_pencilan)]:
    km = KMeans(2, n_init=10, random_state=0).fit(data)
    D = pairwise_distances(data, metric="manhattan")
    med, _ = kmedoids(D, 2)
    print(f"{nama}:")
    print("  pusat k-means :", np.round(sorted(km.cluster_centers_[:, 0]), 2))
    print("  medoid        :", np.round(sorted(data[med, 0]), 2))

## Listing 15.4: Memilih jumlah klaster anggota

`kode/bab15/pilih_k.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab15')
from sklearn.cluster import KMeans
from sklearn.metrics import (calinski_harabasz_score, davies_bouldin_score,
                             silhouette_score)
from sklearn.preprocessing import StandardScaler
from siapkan import muat

df, X = muat()
Z = StandardScaler().fit_transform(X)
print(" k   inersia  silhouette  Calinski-H  Davies-B")
with open("gambar/bab15/pilih_k.dat", "w") as f:
    f.write("k inersia siluet\n")
    for k in range(2, 10):
        km = KMeans(k, n_init=10, random_state=0).fit(Z)
        s = silhouette_score(Z, km.labels_, sample_size=4000,
                             random_state=0)
        ch = calinski_harabasz_score(Z, km.labels_)
        db = davies_bouldin_score(Z, km.labels_)
        print(f"{k:2d} {km.inertia_:9.0f} {s:11.3f} {ch:11.0f} {db:9.3f}")
        f.write(f"{k} {km.inertia_:.0f} {s:.4f}\n")

## Listing 15.5: DBSCAN pada data bulan sabit dan data anggota

`kode/bab15/dbscan.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab15')
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from siapkan import muat

rng = np.random.default_rng(0)
X, y = make_moons(400, noise=0.07, random_state=0)
X = np.vstack([X, rng.uniform([-1.5, -1], [2.5, 1.5], (20, 2))])
y = np.r_[y, -np.ones(20, int)]
km = KMeans(2, n_init=10, random_state=0).fit(X)
db = DBSCAN(eps=0.15, min_samples=8).fit(X)
print(f"bulan sabit: ARI k-means {adjusted_rand_score(y, km.labels_):.3f}"
      f", ARI DBSCAN {adjusted_rand_score(y, db.labels_):.3f}")
print(f"  DBSCAN: {db.labels_.max() + 1} klaster, "
      f"{(db.labels_ == -1).sum()} titik derau")
np.savetxt("gambar/bab15/bulan.dat", np.c_[X, km.labels_, db.labels_],
           fmt="%.3f %.3f %d %d", header="x y km db", comments="")
d, _ = NearestNeighbors(n_neighbors=8).fit(X).kneighbors(X)
np.savetxt("gambar/bab15/k_jarak.dat",
           np.c_[np.arange(len(d)), np.sort(d[:, -1])],
           fmt="%d %.4f", header="i d", comments="")

df, Xa = muat()
Z = StandardScaler().fit_transform(Xa)
print("\nanggota (7 atribut terstandar), min_samples = 14:")
for eps in [0.6, 0.7, 0.8, 1.0]:
    lab = DBSCAN(eps=eps, min_samples=14).fit(Z).labels_
    besar = sorted(np.bincount(lab[lab >= 0]), reverse=True)[:3]
    print(f"  eps={eps:.1f}: {lab.max() + 1:2d} klaster, derau "
          f"{(lab == -1).mean():5.1%}, tiga terbesar {[int(b) for b in besar]}")

## Listing 15.6: Studi kasus segmentasi anggota

`kode/bab15/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab15')
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from siapkan import PERILAKU, muat

df, X = muat()
sk = StandardScaler().fit(X)
Z = sk.transform(X)
km = KMeans(5, n_init=20, random_state=0).fit(Z)
df["klaster"] = km.labels_

prof = df.groupby("klaster").agg(
    n=("id_anggota", "size"), frek=("frekuensi_90h", "mean"),
    struk=("rata_nilai_struk", "mean"), pagi=("porsi_pagi", "mean"),
    akhir_pkn=("porsi_akhir_pekan", "mean"), promo=("porsi_promo", "mean"),
    hitam=("porsi_kopi_hitam", "mean"), apl=("porsi_aplikasi", "mean"),
    umur=("umur", "mean"), churn=("churn", "mean"))
prof["struk"] = prof["struk"] / 1000
print(prof.round(2).to_string())

s = silhouette_samples(Z, km.labels_)
print("\nsilhouette rata-rata per klaster:", "  ".join(
    f"{k}: {v:.3f}" for k, v in pd.Series(s).groupby(km.labels_).mean().items()))

stab = []
rng = np.random.default_rng(0)
for b in range(20):
    i = rng.choice(len(Z), len(Z), replace=True)
    kb = KMeans(5, n_init=5, random_state=b).fit(Z[i])
    stab.append(adjusted_rand_score(km.labels_, kb.predict(Z)))
print(f"stabilitas bootstrap (ARI): rata-rata {np.mean(stab):.3f}, "
      f"minimum {np.min(stab):.3f}")

z = pd.DataFrame(km.cluster_centers_, columns=PERILAKU)
with open("gambar/bab15/pusat_grid.dat", "w") as f:
    for i in range(len(z)):
        for j in range(z.shape[1]):
            f.write(f"{j} {i} {z.iloc[i, j]:.2f}\n")
        f.write("\n")
print("\nsilang dengan tingkat keanggotaan (persen baris):")
print((pd.crosstab(df["klaster"], df["tingkat"], normalize="index")
       * 100).round(0).to_string())
print(f"\nARI terhadap segmen laten (hanya ada di data sintetis): "
      f"{adjusted_rand_score(df['segmen_laten'], km.labels_):.3f}")